# LogReg

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
!pip install datasets huggingface_hub pyarrow

In [ ]:
from datasets import load_dataset

ds = load_dataset("astrosbd/fake-review")
print(ds)

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive/')

df = pd.read_csv('/content/drive/MyDrive/nlp_project/fake_reviews_dataset.csv')
print(df.shape)
print(df.columns.tolist())
print(df['label'].value_counts())

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
(40432, 6)
['category', 'rating', 'label', 'text_', 'user_id', 'date']
label
CG    20216
OR    20216
Name: count, dtype: int64


In [ ]:
df['flagged'] = df['label'].map({'CG': 1, 'OR': 0})
df = df[['text_', 'flagged']].dropna()
df = df.rename(columns={'text_': 'reviewContent'})

X = df['reviewContent']
y = df['flagged']

from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_dev, X_test, y_dev, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)}, Dev: {len(X_dev)}, Test: {len(X_test)}")

Train: 32345, Dev: 4043, Test: 4044


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_dev_tfidf = vectorizer.transform(X_dev)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_dev_pred = model.predict(X_dev_tfidf)
print("--- Dev Set ---")
print(classification_report(y_dev, y_dev_pred))

y_test_pred = model.predict(X_test_tfidf)
print("--- Test Set ---")
print(classification_report(y_test, y_test_pred))

--- Dev Set ---
              precision    recall  f1-score   support

           0       0.92      0.92      0.92      2001
           1       0.92      0.92      0.92      2042

    accuracy                           0.92      4043
   macro avg       0.92      0.92      0.92      4043
weighted avg       0.92      0.92      0.92      4043

--- Test Set ---
              precision    recall  f1-score   support

           0       0.92      0.93      0.93      2070
           1       0.92      0.92      0.92      1974

    accuracy                           0.92      4044
   macro avg       0.92      0.92      0.92      4044
weighted avg       0.92      0.92      0.92      4044



In [ ]:
df_ott = pd.read_csv('/content/drive/MyDrive/nlp_project/deceptive-opinion.csv')
df_ott['label'] = df_ott['deceptive'].map({'deceptive': 1, 'truthful': 0})
X_ott = df_ott['text']
y_ott = df_ott['label']

print("--- Cross-Domain LR (Amazon → Ott) ---")
print(classification_report(y_ott, model.predict(vectorizer.transform(X_ott))))

df_yelp = pd.read_csv(
    '/content/drive/MyDrive/nlp_project/yelp_dataset/new_data_train.csv',
    sep='\t', on_bad_lines='skip', engine='python'
)
df_yelp = df_yelp[['reviewContent', 'flagged']].dropna()
X_yelp = df_yelp['reviewContent']
y_yelp = df_yelp['flagged']

print("--- Cross-Domain LR (Amazon → YelpCHI) ---")
print(classification_report(y_yelp, model.predict(vectorizer.transform(X_yelp))))

--- Cross-Domain LR (Amazon → Ott) ---
              precision    recall  f1-score   support

           0       0.50      0.99      0.66       800
           1       0.38      0.00      0.01       800

    accuracy                           0.50      1600
   macro avg       0.44      0.50      0.34      1600
weighted avg       0.44      0.50      0.34      1600

--- Cross-Domain LR (Amazon → YelpCHI) ---
              precision    recall  f1-score   support

           0       0.51      0.99      0.67      4993
           1       0.65      0.02      0.05      4933

    accuracy                           0.51      9926
   macro avg       0.58      0.51      0.36      9926
weighted avg       0.58      0.51      0.36      9926



In [ ]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_tfidf, y_train)

print("--- Majority Class Baseline → Ott ---")
print(classification_report(y_ott, dummy.predict(vectorizer.transform(X_ott))))

print("--- Majority Class Baseline → YelpCHI ---")
print(classification_report(y_yelp, dummy.predict(vectorizer.transform(X_yelp))))

--- Majority Class Baseline → Ott ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00       800
           1       0.50      1.00      0.67       800

    accuracy                           0.50      1600
   macro avg       0.25      0.50      0.33      1600
weighted avg       0.25      0.50      0.33      1600

--- Majority Class Baseline → YelpCHI ---
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      4993
           1       0.50      1.00      0.66      4933

    accuracy                           0.50      9926
   macro avg       0.25      0.50      0.33      9926
weighted avg       0.25      0.50      0.33      9926



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# BiLSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import numpy as np

In [ ]:
# Tokenize and build vocab from training data only
def tokenize(text):
    return str(text).lower().split()

# Build vocabulary
all_tokens = []
for text in X_train:
    all_tokens.extend(tokenize(text))

vocab_counter = Counter(all_tokens)
vocab = {word: idx+2 for idx, (word, _) in enumerate(vocab_counter.most_common(20000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"Vocab size: {len(vocab)}")

Vocab size: 20002


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=200):
        self.data = []
        for text, label in zip(texts, labels):
            tokens = tokenize(text)[:max_len]
            ids = [vocab.get(t, 1) for t in tokens]
            self.data.append((torch.tensor(ids, dtype=torch.long), int(label)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    return texts_padded, torch.tensor(labels, dtype=torch.long)

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 2)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(emb)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

In [ ]:


# Create datasets
train_dataset = ReviewDataset(X_train, y_train, vocab)
dev_dataset = ReviewDataset(X_dev, y_dev, vocab)
test_dataset = ReviewDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_bilstm = BiLSTMClassifier(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.Adam(model_bilstm.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(3):
    model_bilstm.train()
    total_loss = 0
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_bilstm(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3 - Loss: {total_loss/len(train_loader):.4f}")

Using device: cuda
Epoch 1/3 - Loss: 0.4630
Epoch 2/3 - Loss: 0.2288
Epoch 3/3 - Loss: 0.1514


In [ ]:
from sklearn.metrics import classification_report

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in loader:
            texts = texts.to(device)
            outputs = model(texts)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return all_preds, all_labels

print("--- Dev Set ---")
preds, labels = evaluate(model_bilstm, dev_loader)
print(classification_report(labels, preds))

print("--- Test Set ---")
preds, labels = evaluate(model_bilstm, test_loader)
print(classification_report(labels, preds))

--- Dev Set ---
              precision    recall  f1-score   support

           0       0.97      0.87      0.92      2001
           1       0.89      0.97      0.93      2042

    accuracy                           0.92      4043
   macro avg       0.93      0.92      0.92      4043
weighted avg       0.93      0.92      0.92      4043

--- Test Set ---
              precision    recall  f1-score   support

           0       0.97      0.89      0.93      2070
           1       0.89      0.97      0.93      1974

    accuracy                           0.93      4044
   macro avg       0.93      0.93      0.93      4044
weighted avg       0.93      0.93      0.93      4044



In [ ]:
# Cross-domain evaluation for BiLSTM

def predict_bilstm(model, texts, vocab, device, batch_size=32):
    dataset = ReviewDataset(texts.reset_index(drop=True),
                            [0]*len(texts),  # dummy labels
                            vocab)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)
    model.eval()
    all_preds = []
    with torch.no_grad():
        for text_batch, _ in loader:
            text_batch = text_batch.to(device)
            outputs = model_bilstm(text_batch)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
    return all_preds

print("--- Cross-Domain BiLSTM (Amazon → Ott) ---")
ott_preds = predict_bilstm(model_bilstm, X_ott, vocab, device)
print(classification_report(y_ott.tolist(), ott_preds))

print("--- Cross-Domain BiLSTM (Amazon → YelpCHI) ---")
yelp_preds = predict_bilstm(model_bilstm, X_yelp, vocab, device)
print(classification_report(y_yelp.tolist(), yelp_preds))

--- Cross-Domain BiLSTM (Amazon → Ott) ---
              precision    recall  f1-score   support

           0       0.50      0.99      0.67       800
           1       0.67      0.02      0.03       800

    accuracy                           0.50      1600
   macro avg       0.58      0.50      0.35      1600
weighted avg       0.58      0.50      0.35      1600

--- Cross-Domain BiLSTM (Amazon → YelpCHI) ---
              precision    recall  f1-score   support

           0       0.50      1.00      0.67      4993
           1       0.67      0.01      0.01      4933

    accuracy                           0.50      9926
   macro avg       0.59      0.50      0.34      9926
weighted avg       0.58      0.50      0.34      9926



# BERT

In [ ]:
# Install transformers
!pip install transformers -q

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_len, return_tensors='pt'
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("Creating BERT datasets (this takes a minute)...")
train_bert = BertDataset(X_train, y_train, tokenizer)
dev_bert = BertDataset(X_dev, y_dev, tokenizer)
test_bert = BertDataset(X_test, y_test, tokenizer)

train_bert_loader = DataLoader(train_bert, batch_size=16, shuffle=True)
dev_bert_loader = DataLoader(dev_bert, batch_size=16)
test_bert_loader = DataLoader(test_bert, batch_size=16)
print("Done!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Creating BERT datasets (this takes a minute)...
Done!


In [ ]:
model_bert = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
).to(device)

optimizer_bert = AdamW(model_bert.parameters(), lr=2e-5)

for epoch in range(3):
    model_bert.train()
    total_loss = 0
    for batch in train_bert_loader:
        optimizer_bert.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model_bert(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer_bert.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3 - Loss: {total_loss/len(train_bert_loader):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 - Loss: 0.1112
Epoch 2/3 - Loss: 0.0381
Epoch 3/3 - Loss: 0.0219


In [ ]:
# Evaluate BERT in-domain
def evaluate_bert(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            outputs = model_bert(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return all_preds, all_labels

print("--- BERT Dev Set ---")
preds, labels = evaluate_bert(model_bert, dev_bert_loader, device)
print(classification_report(labels, preds))

print("--- BERT Test Set ---")
preds, labels = evaluate_bert(model_bert, test_bert_loader, device)
print(classification_report(labels, preds))

--- BERT Dev Set ---
              precision    recall  f1-score   support

           0       0.99      0.94      0.97      2001
           1       0.95      0.99      0.97      2042

    accuracy                           0.97      4043
   macro avg       0.97      0.97      0.97      4043
weighted avg       0.97      0.97      0.97      4043

--- BERT Test Set ---
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      2070
           1       0.94      1.00      0.97      1974

    accuracy                           0.97      4044
   macro avg       0.97      0.97      0.97      4044
weighted avg       0.97      0.97      0.97      4044



In [ ]:
def predict_bert_crossdomain(texts, tokenizer, model, device, batch_size=16):
    model.eval()
    all_preds = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        encodings = tokenizer(batch_texts, truncation=True, padding=True,
                              max_length=128, return_tensors='pt')
        input_ids = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)
        with torch.no_grad():
            outputs = model_bert(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
    return all_preds

print("--- Cross-Domain BERT (Amazon → Ott) ---")
ott_preds_bert = predict_bert_crossdomain(X_ott, tokenizer, model_bert, device)
print(classification_report(y_ott.tolist(), ott_preds_bert))

print("--- Cross-Domain BERT (Amazon → YelpCHI) ---")
yelp_preds_bert = predict_bert_crossdomain(X_yelp, tokenizer, model_bert, device)
print(classification_report(y_yelp.tolist(), yelp_preds_bert))

--- Cross-Domain BERT (Amazon → Ott) ---
              precision    recall  f1-score   support

           0       0.50      0.99      0.67       800
           1       0.75      0.02      0.04       800

    accuracy                           0.51      1600
   macro avg       0.63      0.51      0.36      1600
weighted avg       0.63      0.51      0.36      1600

--- Cross-Domain BERT (Amazon → YelpCHI) ---
              precision    recall  f1-score   support

           0       0.50      1.00      0.67      4993
           1       0.53      0.00      0.01      4933

    accuracy                           0.50      9926
   macro avg       0.51      0.50      0.34      9926
weighted avg       0.51      0.50      0.34      9926

